In [ ]:
!pip install torch==2.3.1
!pip install transformers==4.42.3
!pip install trl==0.9.6
!pip install peft==0.11.1
!pip install accelerate==0.32.1
!pip install bitsandbytes==0.43.1
!pip install datasets==2.20.0
!pip install flash-attn==2.6.1 --no-build-isolation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.2/779.2 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 8.0 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.16.0
    Uninstalling peft-0.16.0:
      Successfully uninstalled peft-0.16.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.1/314.1 kB 9.4 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.8.1
    Uninstalling accelerate-1.8.1:
      Successfully uninstalled accelerate-1.8.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 25.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Su

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig, get_peft_model
from datasets import load_dataset, Dataset
import re
from tqdm import tqdm
import os
import zipfile


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
gsm8k = load_dataset("gsm8k", "main")
# My Laptop does not have GPU and in collab it was not working properly
#Please ensure that you are training it heavily as this step is the most vital step in this file
train_data = gsm8k["train"].shuffle(seed=42).select(range(10))  # As My Laptop does not have GPU and in collab it was not working properly I Reduced from 100
test_data = gsm8k["test"].shuffle(seed=42).select(range(10))    #As My Laptop does not have GPU and in collab it was not working properly I Reduced from 50

def extract_ground_truth(answer):
    match = re.search(r"\\boxed\{(.*?)\}", answer)
    return match.group(1) if match else None


train_data = train_data.map(lambda x: {"gt_final": extract_ground_truth(x["answer"])})
test_data = test_data.map(lambda x: {"gt_final": extract_ground_truth(x["answer"])})

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [ ]:
base_model_name = "EleutherAI/gpt-neo-1.3B"
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

tokenizer_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.31G [00:00<?, ?B/s]

In [ ]:
reward_model_name = "microsoft/Phi-3-mini-4k-instruct"
reward_tokenizer = AutoTokenizer.from_pretrained(reward_model_name)

quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

reward_model = AutoModelForCausalLM.from_pretrained(
    reward_model_name,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,

)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [ ]:
def generate_response(model, tokenizer, prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)

def extract_final_answer(response):
    match = re.search(r"\\boxed\{(.*?)\}", response)
    if match:
        return match.group(1)
    numbers = re.findall(r"\d+", response)
    return numbers[-1] if numbers else None

def split_into_steps(response):
    steps = [s.strip() for s in response.split("\n") if s.strip()]
    return steps if steps else [response]

def is_step_correct(question, previous, step):
    judge_prompt = f"Problem: {question}\nPrevious steps: {previous}\nNext step: {step}\nIs this step correct? Answer with Yes or No."
    inputs = reward_tokenizer(judge_prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = reward_model(**inputs).logits[:, -1, :]
    probs = torch.softmax(logits, dim=-1)
    yes_token = reward_tokenizer.convert_tokens_to_ids("Yes")
    no_token = reward_tokenizer.convert_tokens_to_ids("No")
    prob_yes = probs[0, yes_token].item()
    prob_no = probs[0, no_token].item()
    return prob_yes > prob_no, prob_yes

In [ ]:
preference_pairs = []

for example in tqdm(train_data):
    question = example["question"]
    gt_final = example["gt_final"]
    prompt = f"{question}\nLet's think step by step.\n"


    wrong_responses = []
    for _ in range(3):
        response = generate_response(model, tokenizer, prompt)
        pred_final = extract_final_answer(response)
        if pred_final != gt_final:
            wrong_responses.append(response)

    for wrong_resp in wrong_responses:
        steps = split_into_steps(wrong_resp)
        previous = ""
        for i, step in enumerate(steps):
            correct, score = is_step_correct(question, previous, step)
            if not correct:
                s_lose = step

                found_win = False
                for _ in range(3):
                    cont_prompt = f"{question}\n{previous}"
                    cont_response = generate_response(model, tokenizer, cont_prompt, max_new_tokens=300)
                    cont_final = extract_final_answer(cont_response)
                    if cont_final == gt_final:
                        cont_steps = split_into_steps(cont_response)
                        if cont_steps:
                            s_win = cont_steps[0]
                            found_win = True
                            break
                if found_win:
                    pair_prompt = f"{question}\n{previous}"
                    preference_pairs.append({
                        "prompt": pair_prompt,
                        "chosen": s_win,
                        "rejected": s_lose
                    })
                break
            previous += step + "\n"


preference_dataset = Dataset.from_list(preference_pairs)
preference_dataset.save_to_disk("step_dpo_data")
print(f"Generated {len(preference_pairs)} step-wise pairs.")

100%|██████████| 10/10 [09:16<00:00, 55.65s/it]


Saving the dataset (0/1 shards):   0%|          | 0/2 [00:00<?, ? examples/s]

Generated 2 step-wise pairs.


In [ ]:
from trl import DPOTrainer  # Ensure base import if not already

class StepwiseDPOTrainer(DPOTrainer):
    def get_batch_loss_metrics(self, model, batch, train_eval="train"):

        loss, metrics = super().get_batch_loss_metrics(model, batch, train_eval=train_eval)


        metrics["step_avg_loss"] = loss / batch["prompt_input_ids"].shape[0]

        return loss, metrics

In [ ]:

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["c_attn", "c_proj", "c_fc"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

training_args = DPOConfig(
    output_dir="step_dpo_model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    learning_rate=5e-5,
    beta=0.1,
    max_length=512,
    max_prompt_length=256,
    optim="adamw_torch",
    save_strategy="no",
    remove_unused_columns=False,
    report_to="none",
)


trainer = StepwiseDPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=preference_dataset,
    tokenizer=tokenizer,
    peft_config=lora_config,
)


trainer.train()


trainer.save_model("step_dpo_model")
tokenizer.save_pretrained("step_dpo_model")

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss


('step_dpo_model/tokenizer_config.json',
 'step_dpo_model/special_tokens_map.json',
 'step_dpo_model/vocab.json',
 'step_dpo_model/merges.txt',
 'step_dpo_model/added_tokens.json',
 'step_dpo_model/tokenizer.json')

In [ ]:
def evaluate_model(model, tokenizer, test_data):
    correct = 0
    for example in tqdm(test_data):
        prompt = f"{example['question']}\nLet's think step by step.\n"
        response = generate_response(model, tokenizer, prompt, max_new_tokens=200)
        pred_final = extract_final_answer(response)
        if pred_final == example["gt_final"]:
            correct += 1
    return correct / len(test_data)


base_acc = evaluate_model(model, tokenizer, test_data)
print(f"Base Accuracy: {base_acc:.2f}")

from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
trained_model = PeftModel.from_pretrained(
    base_model,
    "step_dpo_model",
    device_map="auto",
)
trained_acc = evaluate_model(trained_model, tokenizer, test_data)
print(f"Trained Accuracy: {trained_acc:.2f}")


with open("results.txt", "w") as f:
    f.write(f"Base: {base_acc}\nTrained: {trained_acc}\n")

100%|██████████| 10/10 [01:07<00:00,  6.73s/it]


Base Accuracy: 0.30


100%|██████████| 10/10 [01:06<00:00,  6.61s/it]

Trained Accuracy: 0.10


In [ ]:
def zip_folder(folder_path, zip_path):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(folder_path):
            for file in files:
                zipf.write(os.path.join(root, file), os.path.relpath(os.path.join(root, file), folder_path))

zip_folder("step_dpo_model", "step_dpo_model.zip")
zip_folder("step_dpo_data", "step_dpo_data.zip")

from google.colab import files
files.download("step_dpo_model.zip")
files.download("step_dpo_data.zip")
files.download("results.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>